# Pipeline demo: detection, tracking, and pitch calibration

*Using Projective Transformation for the Spatial Analysis of Team Behaviors in Football*

This notebook exercises the core of the system built for the thesis:

1. Clone the `pitchvision` package and install its dependencies.
2. Mount Google Drive and locate footage in the `BuildingAction` / `Goals` / `SetPieces` folders.
3. Sanity-check YOLOv8 detection (players + ball) on a single frame.
4. Calibrate the pitch homography from a handful of manually identified pitch landmarks.
5. Run the full detection + ByteTrack tracking + homography pipeline over a clip.
6. Save the resulting per-frame pitch-coordinate tracks to Drive for the phase-specific analyses (goal-scoring opportunity, build-up, set pieces) built in later notebooks.

## 1. Clone the repository and install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/Batomet/Magisterka.git"
BRANCH = "claude/football-spatial-analysis-thesis-lx13x7"
REPO_DIR = "/content/Magisterka"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull

In [ ]:
!pip install -q -e .

## 2. Mount Google Drive and locate footage

In [ ]:
from pitchvision import DriveConfig, mount_drive

mount_drive()

# Adjust `root` if BuildingAction/Goals/SetPieces don't live directly under My Drive.
drive_cfg = DriveConfig(root="/content/drive/MyDrive")
print(drive_cfg.building_action_path)
print(drive_cfg.goals_path)
print(drive_cfg.set_pieces_path)

In [ ]:
from pitchvision import list_videos

goal_videos = list_videos(drive_cfg.goals_path)
print(f"Found {len(goal_videos)} videos in Goals/")

sample_video = goal_videos[0]
sample_video

## 3. Detection sanity check

Runs a COCO-pretrained YOLOv8 model on the first frame, restricted to the `person` (players/referees) and `sports ball` classes.

In [ ]:
import cv2
import matplotlib.pyplot as plt

from pitchvision import PlayerBallDetector, VideoFrames

frames = VideoFrames(sample_video)
first_frame = frames.read_frame(0)

detector = PlayerBallDetector(weights="yolov8n.pt", confidence=0.3)
detections = detector.detect(first_frame)
print(f"{len(detections)} detections in frame 0")

vis = first_frame.copy()
for det in detections:
    x1, y1, x2, y2 = det.xyxy.astype(int)
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(vis, det.class_name, (x1, max(y1 - 5, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("YOLOv8 detections (person + sports ball)")
plt.show()

## 4. Pitch calibration

Identify at least 4 pitch landmarks that are clearly visible in the frame above (corner flags, penalty-box corners, six-yard-box corners, the centre circle, the halfway line intersections, ...). Zoom into the plot to read off pixel coordinates, then fill in `landmark_pixels` below, matching each pixel position to the correct entry in `PITCH_LANDMARKS_M`.

More correspondences, spread across the frame rather than clustered in one corner, give a more accurate homography.

In [ ]:
from pitchvision import PITCH_LANDMARKS_M

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
plt.title("Zoom in (use the toolbar) and read pixel coordinates for the landmarks below")
plt.show()

print("Available landmark names:", list(PITCH_LANDMARKS_M.keys()))

In [ ]:
import numpy as np

from pitchvision import PitchCalibrator

# EDIT THIS: pixel (x, y) coordinates for each landmark you identified above.
landmark_pixels = {
    "left_penalty_top": (123, 456),
    "left_penalty_bottom": (110, 620),
    "left_six_yard_top": (250, 430),
    "centre_spot": (900, 300),
}

pixel_points = np.array(list(landmark_pixels.values()))
pitch_points = np.array([PITCH_LANDMARKS_M[name] for name in landmark_pixels])

calibrator = PitchCalibrator.from_point_pairs(pixel_points, pitch_points)
print("Mean reprojection error (m):", calibrator.reprojection_error(pixel_points, pitch_points))

### Sanity-check the calibration

Project the frame's detected players onto the top-down pitch view; they should land inside the pitch outline in positions that roughly match the original frame.

In [ ]:
from pitchvision import draw_pitch, plot_positions

foot_points = np.array([d.foot_point for d in detections if d.class_name == "person"])
pitch_positions = calibrator.pixel_to_pitch(foot_points)

ax = draw_pitch()
plot_positions(ax, pitch_positions, color="yellow", s=40, edgecolors="black")
plt.title("Detected players projected onto the pitch (frame 0)")
plt.show()

## 5. Full pipeline: detection + ByteTrack tracking + homography

`max_frames` caps the run while iterating on calibration; set it to `None` for the full clip.

In [ ]:
from pitchvision import PlayerTracker, TrackingPipeline

tracker = PlayerTracker(weights="yolov8n.pt", confidence=0.3)
pipeline = TrackingPipeline(tracker=tracker, calibrator=calibrator)

tracks_df = pipeline.run(sample_video, max_frames=250)
tracks_df.head()

In [ ]:
ax = draw_pitch()
for track_id, group in tracks_df[tracks_df["class_name"] == "person"].groupby("track_id"):
    ax.plot(group["pitch_x"], group["pitch_y"], linewidth=1, alpha=0.7)
plt.title("Player trajectories over the sampled frames")
plt.show()

## 6. Save tracks for downstream analysis

In [ ]:
output_dir = "/content/drive/MyDrive/pitchvision_outputs"
os.makedirs(output_dir, exist_ok=True)

out_path = os.path.join(output_dir, os.path.splitext(os.path.basename(sample_video))[0] + "_tracks.csv")
tracks_df.to_csv(out_path, index=False)
out_path

## Next steps

This notebook covers the shared foundation only. Not yet implemented, planned for later notebooks:

- **Team classification** (e.g. jersey-colour clustering) to split tracks into the two teams, needed by every downstream analysis.
- **Goal-scoring opportunity phase**: defensive compactness (inter-player distances) and space control (Voronoi diagrams).
- **Build-up phase**: Effective Playing Space via Convex Hull, and stretch/compactness via formation centroids.
- **Set pieces / breaks in play**: repeatability of positional structures, and static-to-dynamic transition timing after the restart.